# Lab 1: from a base model to an assistant

**RLHF course, session 1 · Pegah Alizadeh**

This lab follows Chapter 4 of Nathan Lambert's [RLHF book](https://rlhfbook.com) and is adapted from the book's
[`code/instruction_tuning`](https://github.com/natolambert/rlhf-book/tree/main/code/instruction_tuning) example.

**Goal:** see the main effect of supervised fine-tuning (SFT): *a base model continues text; an instruction-tuned
model answers and stops.*

| Part | What you do | Time |
|---|---|---|
| 1 | Run the base model | 10 min |
| 2 | Read the chat template and write the prompt mask (exercise A) | 10 min |
| 3 | Compute the response log-probability by hand (exercise B) | 15 min |
| 4 | Train on a small slice | 10 min (it runs while you discuss) |
| 5 | Compare before and after, with uncertainty (exercise C) | 20 min |

If you get stuck, each exercise has a hint and a solution you can open. The aim is to understand the steps, so use them.

## Before you start

- **Colab:** Runtime → Change runtime type → **T4 GPU**.
- **Kaggle:** Settings → Accelerator → **GPU T4 x2**. Turn on **Internet** in the same panel.
- If you have no GPU, ask to pair with a neighbor.
- **Out of memory?** Restart the session (Colab: Runtime → Restart session; Kaggle: Run → Restart & clear cell outputs), then run all the cells again from the top.

In [ ]:
# Colab and Kaggle already include PyTorch. This adds or updates the other libraries.
%pip install -q -U transformers datasets accelerate

In [ ]:
import gc
import math
import random
import time

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (training will be very slow)")

In [ ]:
# Everything you may want to change later is here.
MODEL_NAME = "allenai/OLMo-2-0425-1B"            # a 1B base model
TEMPLATE_SOURCE = "allenai/OLMo-2-0425-1B-SFT"   # the base tokenizer has no chat template
DATASET_NAME = "HuggingFaceH4/no_robots"         # human-written instructions, CC BY-NC 4.0

N_TRAIN = 3000       # the book's full run uses about 9,500 rows
MAX_LENGTH = 256     # the book's full run uses 2048; shorter examples train much faster
BATCH_SIZE = 4
ACCUM_STEPS = 4      # effective batch size = 4 x 4 = 16
EPOCHS = 2
LR = 1e-5
TRAIN_LAST_LAYERS = 4   # of 16: the rest stay frozen, so that training fits on a free T4
MAX_MINUTES = 12     # training stops cleanly at this limit

N_EVAL = 32          # held-out prompts for Part 5
MAX_NEW_TOKENS = 96
SEED = 42
IGNORE_INDEX = -100

random.seed(SEED)
torch.manual_seed(SEED)

## Part 1: meet the base model

Downloading the model takes a minute or two. It has only been through pretraining: it was trained to predict the next
token on raw text and has never been taught to answer anyone.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.chat_template = AutoTokenizer.from_pretrained(TEMPLATE_SOURCE).chat_template
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # generation continues from the right-hand end
tokenizer.truncation_side = "left"   # very long prompts keep their end, including the cue to answer

# If this cell is run again, remove the old model and training state first, so that two copies don't fill the GPU.
for name in ["model", "optimizer", "scheduler", "scaler", "logits", "loss", "input_ids", "attention", "labels", "total"]:
    globals().pop(name, None)
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# On a T4, fp16 is many times faster than bf16 (see the note in Part 4).
dtype = torch.float16 if device == "cuda" else torch.bfloat16
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters, dtype {dtype}")

In [ ]:
@torch.no_grad()
def generate(prompts, use_template=True, max_new_tokens=MAX_NEW_TOKENS, batch_size=8):
    """Greedy generation. Returns the response text and whether the model stopped by itself."""
    model.eval()
    model.config.use_cache = True
    if use_template:
        texts = [
            tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
            for p in prompts
        ]
    else:
        texts = list(prompts)

    responses, stopped = [], []
    for start in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[start:start + batch_size], return_tensors="pt", padding=True, truncation=True,
            max_length=MAX_LENGTH, add_special_tokens=False,
        ).to(device)
        with torch.amp.autocast(device, dtype=dtype):
            out = model.generate(
                **enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        new_tokens = out[:, enc["input_ids"].shape[1]:]
        for row in new_tokens:
            has_eos = (row == tokenizer.eos_token_id).any().item()
            responses.append(tokenizer.decode(row, skip_special_tokens=True).strip())
            stopped.append(bool(has_eos))
    return responses, stopped


DEMO_PROMPTS = [
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
    "Write a haiku about programming.",
    "How does photosynthesis work?",
]

before_demo, _ = generate(DEMO_PROMPTS)
for prompt, response in zip(DEMO_PROMPTS, before_demo):
    print(f"PROMPT: {prompt}\nBASE MODEL: {response[:400]}\n{'-' * 70}")

**Discuss with your neighbor:** does the model know any of the answers? What does it do wrong? The book's claim is that
much of the knowledge is already here, and SFT mostly teaches the format. Keep that in mind for Part 5.

## Part 2: chat templates and the prompt mask

A chat template turns a list of messages into one flat text. The model only ever sees that text.

In [ ]:
dataset = load_dataset(DATASET_NAME)
example = dataset["train"][0]["messages"]

print(tokenizer.apply_chat_template(example, tokenize=False))
print("=" * 70)
print("Prompt only, with the cue to start answering:")
print(tokenizer.apply_chat_template(example[:-1], tokenize=False, add_generation_prompt=True))

Compare the two outputs above. Where does the response begin, and which token ends it?

### Exercise A: write the labels

In SFT the prompt **conditions** the prediction, but only response tokens contribute to the loss. PyTorch's cross-entropy
ignores any position whose label is `-100`.

Complete `build_labels`. It receives the token ids of the prompt and of the full conversation.

<details><summary>Hint</summary>

The prompt is a prefix of the full conversation, so the first `len(prompt_ids)` labels are `IGNORE_INDEX`.
</details>

<details><summary>Solution</summary>

```python
def build_labels(prompt_ids, full_ids):
    n_prompt = len(prompt_ids)
    return [IGNORE_INDEX] * n_prompt + full_ids[n_prompt:]
```
</details>

In [ ]:
def build_labels(prompt_ids, full_ids):
    # TODO: return a list as long as full_ids, with IGNORE_INDEX on every
    # prompt position and the true token id on every response position.
    raise NotImplementedError("Complete exercise A")

In [ ]:
# Check your answer
assert build_labels([1, 2], [1, 2, 3, 4]) == [IGNORE_INDEX, IGNORE_INDEX, 3, 4]
assert build_labels([], [7]) == [7]
print("Exercise A: correct")

In [ ]:
def encode(messages):
    """Tokenize one conversation and mask everything except the final assistant turn."""
    if not messages or messages[-1]["role"] != "assistant":
        return None
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    # Rows that do not fit are dropped, not cut, so that every example still ends with the stop token.
    if len(full_ids) > MAX_LENGTH or full_ids[:len(prompt_ids)] != prompt_ids or len(full_ids) == len(prompt_ids):
        return None
    return {"input_ids": full_ids, "labels": build_labels(prompt_ids, full_ids)}


train_rows = dataset["train"].shuffle(seed=SEED).select(range(min(2 * N_TRAIN, len(dataset["train"]))))
train_data = [e for e in (encode(row["messages"]) for row in train_rows) if e is not None][:N_TRAIN]
print(f"{len(train_data)} training examples")

sample = train_data[0]
print("Tokens that receive a loss:")
print(tokenizer.decode([t for t in sample["labels"] if t != IGNORE_INDEX]))

In [ ]:
def collate(examples):
    """Right-pad a batch for training."""
    width = max(len(e["input_ids"]) for e in examples)
    pad_id = tokenizer.pad_token_id
    input_ids = [e["input_ids"] + [pad_id] * (width - len(e["input_ids"])) for e in examples]
    labels = [e["labels"] + [IGNORE_INDEX] * (width - len(e["labels"])) for e in examples]
    attention = [[1] * len(e["input_ids"]) + [0] * (width - len(e["input_ids"])) for e in examples]
    return (
        torch.tensor(input_ids, device=device),
        torch.tensor(attention, device=device),
        torch.tensor(labels, device=device),
    )

## Part 3: the log-probability of a response

This is the lecture's tensor walk-through, in code. It is also the most useful skill for the rest of the course: DPO in
session 3 is built from exactly this quantity, for two responses at a time.

### Exercise B: compute it by hand

The mask comes from the labels: `IGNORE_INDEX` marks the tokens that don't count. (After class, HW0 practises this step by step on tiny tensors.)

Step 1 is done for you. Complete the others in `response_log_prob`, then run the two checks. The first uses a tiny example that you can work out by hand.

1. **Shift the logits:** drop the last position, because it has no next token to predict.
2. **Shift the targets:** drop the first label, so that position $t$ predicts token $t+1$.
3. **Build the mask:** `True` where the target is not `IGNORE_INDEX`.
4. **Read off the log-probabilities:** apply `log_softmax`, then use `gather` to pick each true token. `gather` cannot accept
   `-100`, so use `targets.clamp(min=0)` as the index.
5. **Sum per sequence:** multiply by the mask, then sum. Return the sum and the number of counted tokens.

<details><summary>Hint</summary>

`log_probs.gather(-1, index.unsqueeze(-1)).squeeze(-1)` has shape `(B, L-1)`.
</details>

<details><summary>Solution</summary>

```python
def response_log_prob(logits, labels):
    logits = logits[:, :-1, :]
    targets = labels[:, 1:]
    mask = targets != IGNORE_INDEX
    log_probs = logits.log_softmax(dim=-1)
    token_log_probs = log_probs.gather(-1, targets.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    return (token_log_probs * mask).sum(dim=-1), mask.sum(dim=-1)
```
</details>

In [ ]:
def response_log_prob(logits, labels):
    # logits: (B, L, V)   labels: (B, L), with IGNORE_INDEX on the prompt and padding
    # If you're unsure about a step, add print(name.shape) and compare with the shape in the comment.

    # Step 1 (done for you): drop the last position, which has no next token to predict.
    logits = logits[:, :-1, :]                     # (B, L-1, V)

    # Step 2: drop the first label, so that position t is compared with token t+1.
    targets = None  # TODO                         # (B, L-1)

    # Step 3: True where the target counts in the loss, that is, where it is not IGNORE_INDEX.
    mask = None  # TODO                            # (B, L-1)

    # Step 4: log-probabilities over the vocabulary, then the one of each true next token.
    #         gather can't use -100 as an index, so use targets.clamp(min=0) as the index.
    log_probs = None  # TODO                       # (B, L-1, V)
    token_log_probs = None  # TODO                 # (B, L-1)

    # Step 5: add up the counted positions, and count them.
    total = None  # TODO                           # (B,)
    n_tokens = None  # TODO                        # (B,)
    return total, n_tokens

In [ ]:
# Check 1, on a tiny example you can work out by hand.
# With all-zero scores, every token has probability 1/V, so each counted token adds log(1/V).
V = 7
toy_logits = torch.zeros(2, 5, V)
toy_labels = torch.tensor([[IGNORE_INDEX, IGNORE_INDEX, 3, 1, 4],
                           [IGNORE_INDEX, 2, 2, 0, IGNORE_INDEX]])
total, n_tokens = response_log_prob(toy_logits, toy_labels)
assert total is not None and n_tokens is not None, "Complete exercise B: some steps are still None."
print("Counted tokens:", n_tokens.tolist(), " Log-probabilities:", [round(x, 3) for x in total.tolist()])
assert n_tokens.tolist() == [3, 3], "Each sequence should count 3 tokens. Check the shift (step 2) and the mask (step 3)."
assert torch.allclose(total, torch.full((2,), -3 * math.log(V))), "Expected 3 x log(1/7) = -5.838 for each sequence."
print("Check 1: correct")

In [ ]:
# Check 2, on the real model: the negative mean log-probability must equal the cross-entropy that training uses.
model.eval()
with torch.no_grad():
    input_ids, attention, labels = collate(train_data[:4])
    with torch.amp.autocast(device, dtype=dtype):
        logits = model(input_ids=input_ids, attention_mask=attention, use_cache=False).logits
    logits = logits.float()
    total, n_tokens = response_log_prob(logits, labels)
    mine = -(total.sum() / n_tokens.sum()).item()
    reference = F.cross_entropy(
        logits[:, :-1].reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1), ignore_index=IGNORE_INDEX
    ).item()

print(f"Yours: {mine:.4f}   Cross-entropy: {reference:.4f}")
assert abs(mine - reference) < 1e-3, "These should match. Check the shift and the mask."
print("Exercise B: correct")
print("Log-probability of each response:", [round(x, 1) for x in total.tolist()])

**Discuss:** the log-probabilities above are very negative. Is that a problem? Compare two responses of different lengths:
what does the length do to this number? (This returns in session 3, when we compare DPO with length-normalized variants.)

## Part 4: train on a small slice

First we measure the base model on **held-out** prompts, then we train. Training uses the same cross-entropy as pretraining;
only the data and the mask have changed.

Start the training cell and let it run while you discuss. It stops by itself after `MAX_MINUTES`.

**How we fit this on a free GPU.** This model has about 1.48B parameters. Every trainable parameter also needs a gradient
and two AdamW statistics, which is too much for a T4's 15 GB, so we differ from the book's full run in three ways:

- **Frozen layers:** we train only the last 4 of the 16 layers and the output head.
- **Short examples:** we keep only conversations of at most 256 tokens, so the model sees shorter responses than in the full dataset.
- **Mixed precision:** frozen weights use 16-bit floats, trainable weights use 32 bits, and a *gradient scaler* stops small
  16-bit gradients from rounding to zero. On a T4 this ran about 5 times faster than `bf16`.

In [ ]:
# Evaluate the base model before training, on prompts from the held-out test split.
eval_rows = dataset["test"].shuffle(seed=SEED).select(range(N_EVAL))
eval_prompts = [row["messages"][0]["content"] if row["messages"][0]["role"] == "user" else row["prompt"] for row in eval_rows]

t0 = time.time()
before_eval, before_stopped = generate(eval_prompts)
print(f"Base model stopped by itself on {sum(before_stopped)}/{N_EVAL} prompts ({time.time() - t0:.0f}s)")

In [ ]:
torch.manual_seed(SEED)
random.seed(SEED)
model.train()
model.config.use_cache = False

# Freeze everything except the output head and the last layers (see the note above this cell).
for param in model.parameters():
    param.requires_grad = False
for module in [model.lm_head, model.model.norm, *model.model.layers[-TRAIN_LAST_LAYERS:]]:
    for param in module.parameters():
        param.requires_grad = True
        if dtype == torch.float16:
            param.data = param.data.float()
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Training {sum(p.numel() for p in trainable) / 1e9:.2f}B of {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")

micro_batches_per_epoch = len(train_data) // BATCH_SIZE
total_steps = EPOCHS * micro_batches_per_epoch // ACCUM_STEPS
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0, foreach=False)
scaler = torch.amp.GradScaler(device, enabled=dtype == torch.float16)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

history, running, step = [], 0.0, 0
start = time.time()
finished = False
optimizer.zero_grad()

for epoch in range(EPOCHS):
    order = random.sample(range(len(train_data)), len(train_data))
    for i in range(micro_batches_per_epoch):
        batch = [train_data[j] for j in order[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]]
        input_ids, attention, labels = collate(batch)
        with torch.amp.autocast(device, dtype=dtype):
            logits = model(input_ids=input_ids, attention_mask=attention, use_cache=False).logits
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1), ignore_index=IGNORE_INDEX
            )
        scaler.scale(loss / ACCUM_STEPS).backward()
        running += loss.item() / ACCUM_STEPS

        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step += 1
            history.append(running)
            running = 0.0
            if step % 10 == 0 or step == 1:
                minutes = (time.time() - start) / 60
                print(f"step {step:4d}/{total_steps}  loss {history[-1]:.3f}  elapsed {minutes:5.1f} min")
            if (time.time() - start) / 60 > MAX_MINUTES:
                print("Time limit reached: stopping early, which is fine for this lab.")
                finished = True
                break
    if finished:
        break

minutes = (time.time() - start) / 60
if device == "cuda":
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

# Free the optimizer and gradients, so that generation has enough memory in Part 5.
del optimizer, scheduler, scaler, logits, loss, input_ids, attention, labels
model.zero_grad(set_to_none=True)
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
print(f"Done: {step} optimizer steps in {minutes:.1f} min")

window = 20
smoothed = [sum(history[max(0, i - window + 1):i + 1]) / len(history[max(0, i - window + 1):i + 1]) for i in range(len(history))]
plt.plot(history, alpha=0.3, label="Each step")
plt.plot(smoothed, label=f"Average of last {window} steps")
plt.legend()
plt.xlabel("Optimizer step")
plt.ylabel("Training loss")
plt.title("SFT training loss")
plt.show()

## Part 5: what changed?

The loss above is noisy, because each step sees only 16 examples, and it falls only slowly. Don't judge the training from the loss alone: the change in behavior below is much clearer.

In [ ]:
after_demo, _ = generate(DEMO_PROMPTS)
for prompt, base, tuned in zip(DEMO_PROMPTS, before_demo, after_demo):
    print(f"PROMPT: {prompt}\n\nBEFORE: {base[:300]}\n\nAFTER: {tuned[:300]}\n{'-' * 70}")

In [ ]:
after_eval, after_stopped = generate(eval_prompts)
print(f"Stopped by itself: before {sum(before_stopped)}/{N_EVAL}, after {sum(after_stopped)}/{N_EVAL}")

Reading examples is not enough: we also want a number, with its uncertainty. A simple measure here is the proportion of
held-out prompts where the model **stops by itself** within the token limit.

### Exercise C: an interval for the stop rate

Complete `wilson_interval`. For $k$ successes in $n$ trials, with $\hat p = k/n$:

$$
\frac{\hat p + \frac{z^2}{2n}}{1 + \frac{z^2}{n}} \;\pm\; \frac{z}{1 + \frac{z^2}{n}}\sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}}
$$

<details><summary>Solution</summary>

```python
def wilson_interval(successes, n, z=1.96):
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - half, center + half
```
</details>

In [ ]:
def wilson_interval(successes, n, z=1.96):
    # TODO: return (lower, upper) for the Wilson score interval of a proportion.
    raise NotImplementedError("Complete exercise C")

In [ ]:
# Check your answer
low, high = wilson_interval(8, 10)
assert abs(low - 0.4902) < 1e-3 and abs(high - 0.9433) < 1e-3
print("Exercise C: correct\n")

for name, stops in [("Before", before_stopped), ("After", after_stopped)]:
    low, high = wilson_interval(sum(stops), N_EVAL)
    print(f"{name}: stop rate {sum(stops) / N_EVAL:.2f}, 95% interval [{low:.2f}, {high:.2f}]")

# The same prompts were used twice, so the fair comparison is paired.
only_after = sum((not b) and a for b, a in zip(before_stopped, after_stopped))
only_before = sum(b and (not a) for b, a in zip(before_stopped, after_stopped))
print(f"\nStopped only after training: {only_after}. Stopped only before training: {only_before}.")

In [ ]:
# Read a few held-out responses yourself.
for i in range(min(3, N_EVAL)):
    print(f"PROMPT: {eval_prompts[i][:200]}\n\nBEFORE: {before_eval[i][:300]}\n\nAFTER: {after_eval[i][:300]}\n{'-' * 70}")

**Discuss before leaving:**

1. Did the model learn to stop? Did it also learn new facts? What in these outputs supports your answer?
2. Is the stop rate a good measure of quality? Find a response that stops but is still poor. Find one that is good but was cut off by the token limit, and so counts as "not stopped".
3. The two intervals treat the runs as independent. Why is the paired count above more appropriate?

## Going further (optional)

These are for after class, if you want to go deeper. Change one thing at a time, and reload the model before each new run.

- **Remove the mask:** train on prompt tokens too. What changes, and why? Can you see it in the outputs?
- **Frozen layers:** vary `TRAIN_LAST_LAYERS` (1, 2, 4). How much of the model has to change for it to learn to stop?
- **Dataset size:** vary `N_TRAIN` (500, 1,000, 3,000). How much data does it take for the stop rate to rise?
- **Learning rate:** try `LR` at 5e-6 and 2e-5. What goes wrong, and when?
- **Statistics:** derive an exact test for the paired counts above (McNemar's test). What can you conclude with 32 prompts, and how many would you need to detect a smaller effect?
- **Reading:** Chapter 4, Implementation Details, and the book's [full code](https://github.com/natolambert/rlhf-book/tree/main/code/instruction_tuning).